# Basic Seq2Seq + LSTM

### Imports

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import torch
from nmt.models.encoder import Encoder
from nmt.models.decoder import Decoder
from nmt.models.seq2seq import Seq2Seq


### Load Data

In [13]:
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

# ── path setup (must come before src.* imports) ──────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── imports ───────────────────────────────────────────────────────────────
from src.nmt.data.dataset import TranslationDataset, collate_fn
from src.nmt.tokenization.tokenizer import WordTokenizer
from src.nmt.tokenization.vocabulary import Vocabulary

# ── paths ─────────────────────────────────────────────────────────────────
DATA_DIR  = PROJECT_ROOT / "data" / "processed" / "mt560_amharic_english"
VOCAB_DIR = PROJECT_ROOT / "artifacts" / "vocabularies"

# ── load data ─────────────────────────────────────────────────────────────
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df   = pd.read_parquet(DATA_DIR / "validation.parquet")
test_df  = pd.read_parquet(DATA_DIR / "test.parquet")

en_vocab  = Vocabulary.load(VOCAB_DIR / "en_vocab.json")
am_vocab  = Vocabulary.load(VOCAB_DIR / "am_vocab.json")
tokenizer = WordTokenizer()

BATCH_SIZE = 64

# ── datasets ──────────────────────────────────────────────────────────────
train_dataset = TranslationDataset(train_df, tokenizer, en_vocab, am_vocab)
val_dataset   = TranslationDataset(val_df,   tokenizer, en_vocab, am_vocab)
test_dataset  = TranslationDataset(test_df,  tokenizer, en_vocab, am_vocab)

# ── dataloaders ───────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train      : {len(train_dataset):,}")
print(f"Validation : {len(val_dataset):,}")
print(f"Test       : {len(test_dataset):,}")
print(f"Batch size : {BATCH_SIZE}")


c:\Users\olibe\Desktop\neural-machine-translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train      : 529,992
Validation : 66,249
Test       : 66,250
Batch size : 64


### Device + hyperparameters

In [14]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(am_vocab)

ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HIDDEN_DIM = 512

N_LAYERS = 1
DROPOUT = 0.2

TEACHER_FORCING_RATIO = 0.5

print("Device:", DEVICE)
print("Input vocabulary:", INPUT_DIM)
print("Output vocabulary:", OUTPUT_DIM)

Device: cpu
Input vocabulary: 54871
Output vocabulary: 182658


### Build model

In [15]:
encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=ENC_EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=N_LAYERS,
    dropout=DROPOUT,
)

decoder = Decoder(
    output_dim=OUTPUT_DIM,
    embedding_dim=DEC_EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=N_LAYERS,
    dropout=DROPOUT,
)

model = Seq2Seq(
    encoder,
    decoder,
    DEVICE,
).to(DEVICE)

print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(54871, 256, padding_idx=0)
    (rnn): LSTM(256, 512, batch_first=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(182658, 256, padding_idx=0)
    (rnn): LSTM(256, 512, batch_first=True)
    (fc_out): Linear(in_features=512, out_features=182658, bias=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
)


### Count parameters

In [16]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

Total parameters: 157,664,898
Trainable parameters: 157,664,898


### Forward-pass test

In [17]:
src_batch, tgt_batch = next(iter(train_loader))

src_batch = src_batch.to(DEVICE)
tgt_batch = tgt_batch.to(DEVICE)

with torch.no_grad():
    output = model(
        src_batch,
        tgt_batch,
        teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    )

print("Source shape :", src_batch.shape)
print("Target shape :", tgt_batch.shape)
print("Output shape :", output.shape)

Source shape : torch.Size([64, 64])
Target shape : torch.Size([64, 61])
Output shape : torch.Size([64, 61, 182658])


### Quality Gate

In [18]:
assert output.ndim == 3

assert output.shape[0] == src_batch.shape[0]
assert output.shape[1] == tgt_batch.shape[1]
assert output.shape[2] == OUTPUT_DIM

assert torch.isfinite(output).all()

print("BASELINE SEQ2SEQ QUALITY GATE PASSED")

BASELINE SEQ2SEQ QUALITY GATE PASSED
